# HAM10000 Skin Lesion Classifier — Kaggle Training

**Environment requirements (set before running):**
- Accelerator: **GPU P100** (Settings → Accelerator)
- Internet: **On** (Settings → Internet)
- Dataset attached: `kmader/skin-cancer-mnist-ham10000`

**What this notebook does:**
1. Installs missing packages (timm, albumentations, grad-cam)
2. Clones the project repo from GitHub
3. Symlinks the Kaggle dataset into the expected `data/` path
4. Regenerates train/val/test splits from the Kaggle CSV
5. Runs the full 30-epoch two-stage training (~45–60 min on P100)
6. Copies `best_model.pth` + `last_model.pth` to `/kaggle/working/` for download
7. Runs a sanity check on the saved checkpoint

**After training:** go to the Output tab and download `best_model.pth`.

In [ ]:
# Cell 1 — Install missing packages
# Kaggle already provides: torch (CUDA), torchvision, pandas, scikit-learn,
# tensorboard, tqdm, pillow, opencv-python.
# Only the three packages below need to be added.
!pip install timm==1.0.26 albumentations==2.0.8 "grad-cam==1.5.5" --quiet

# Verify CUDA is available — should print True on P100
import torch
print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2 — Clone repo and wire up the data directory
import os, subprocess

REPO_DIR = "/kaggle/working/repo"
DATASET_DIR = "/kaggle/input/skin-cancer-mnist-ham10000"

# Clone the repo (shallow clone for speed)
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Hans-Randy/skin-lesion-classifier.git",
         REPO_DIR],
        check=True
    )
    print(f"Repo cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}")

# Symlink the Kaggle-managed dataset into the path train.py expects.
# The repo's .gitignore excludes data/ so there's nothing there to overwrite.
data_link = os.path.join(REPO_DIR, "data")
if not os.path.exists(data_link):
    os.symlink(DATASET_DIR, data_link)
    print(f"Symlinked {DATASET_DIR} -> {data_link}")
else:
    print(f"data/ already linked: {os.readlink(data_link) if os.path.islink(data_link) else 'directory'}")

# Quick sanity: confirm the metadata CSV is reachable through the symlink
meta_path = os.path.join(data_link, "HAM10000_metadata.csv")
assert os.path.exists(meta_path), f"Metadata CSV not found at {meta_path}"
print(f"Metadata CSV found: {meta_path}")

In [ ]:
# Cell 3 — Regenerate splits from the Kaggle CSV
# --overwrite ensures the splits/_meta.json MD5 matches the Kaggle-hosted CSV
# (the hash may differ from the one committed to the repo if Kaggle re-encoded
# the file). train.py's validate_existing_splits() checks this hash at startup.
!cd /kaggle/working/repo && python scripts/make_splits.py --overwrite

In [ ]:
# Cell 4 — Full two-stage training (30 epochs)
# Stage 1: 5 epochs, frozen backbone, LR 1e-3  (~3-5 min on P100)
# Stage 2: 25 epochs, full fine-tune, LR 1e-5  (~40-50 min on P100)
# AMP (mixed precision) is enabled automatically when CUDA is available.
# TensorBoard logs -> /kaggle/working/repo/runs/<timestamp>/
# Checkpoints     -> /kaggle/working/repo/checkpoints/
!cd /kaggle/working/repo && python train.py \
    --epochs 30 \
    --batch-size 32 \
    --lr 1e-5 \
    --warmup-epochs 5 \
    --warmup-lr 1e-3 \
    --num-workers 4 \
    --seed 42

In [ ]:
# Cell 5 — Copy checkpoints to /kaggle/working/ for download
import shutil, os

CKPT_SRC = "/kaggle/working/repo/checkpoints"
CKPT_DST = "/kaggle/working"

for fname in ["best_model.pth", "last_model.pth"]:
    src = os.path.join(CKPT_SRC, fname)
    dst = os.path.join(CKPT_DST, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f"Copied {fname} ({size_mb:.1f} MB) -> {dst}")
    else:
        print(f"WARNING: {src} not found — training may have failed")

print("\nOutput tab files:")
for f in os.listdir(CKPT_DST):
    if f.endswith(".pth"):
        print(f"  {f}  ({os.path.getsize(os.path.join(CKPT_DST, f)) / 1e6:.1f} MB)")

In [ ]:
# Cell 6 — Sanity check
# Verifies: class list matches CLASSES, output shape (1,7), probs sum to 1.
!cd /kaggle/working/repo && python scripts/sanity_check.py \
    --checkpoint /kaggle/working/best_model.pth

print("\n=" * 30)
print("All done! Download best_model.pth from the Output tab.")
print("Then run locally:")
print("  uv run python evaluate.py --checkpoint checkpoints/best_model.pth")
print("  uv run python app.py --checkpoint checkpoints/best_model.pth")